In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV,LinearRegression, Lasso, Ridge,MultiTaskLassoCV,MultiTaskLasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler,KBinsDiscretizer, PolynomialFeatures
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import f_regression
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import SelectFromModel


In [2]:
#1. Load data 
price_train= pd.read_csv("price_train_processed.csv")
price_test= pd.read_csv("price_test_processed.csv")
rent_train= pd.read_csv("rent_train_processed.csv")
rent_test= pd.read_csv("rent_test_processed.csv")

In [3]:
price_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103871 entries, 0 to 103870
Data columns (total 60 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   城市         103871 non-null  int64  
 1   区域         103871 non-null  float64
 2   板块         103871 non-null  float64
 3   环线         103871 non-null  int64  
 4   房屋户型       103291 non-null  object 
 5   所在楼层       103871 non-null  object 
 6   建筑面积       103871 non-null  float64
 7   套内面积       103871 non-null  float64
 8   建筑结构       103871 non-null  int64  
 9   装修情况       103871 non-null  int64  
 10  梯户比例       103871 non-null  float64
 11  配备电梯       103871 non-null  int64  
 12  别墅类型       103871 non-null  int64  
 13  交易时间       103871 non-null  object 
 14  交易权属       103871 non-null  int64  
 15  上次交易       78405 non-null   object 
 16  房屋用途       103871 non-null  int64  
 17  房屋年限       103871 non-null  int64  
 18  产权所属       103871 non-null  int64  
 19  lon        103871 non-n

In [4]:
rent_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 57 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      98899 non-null  int64  
 1   户型      98898 non-null  object 
 2   装修      98899 non-null  float64
 3   楼层      98899 non-null  float64
 4   面积      98899 non-null  float64
 5   交易时间    98899 non-null  object 
 6   付款方式    98899 non-null  int64  
 7   租赁方式    98899 non-null  int64  
 8   电梯      98899 non-null  float64
 9   车位      98899 non-null  int64  
 10  用水      90203 non-null  object 
 11  用电      90410 non-null  object 
 12  燃气      98899 non-null  float64
 13  采暖      40992 non-null  object 
 14  租期      98899 non-null  int64  
 15  配套设施    68448 non-null  object 
 16  lon     98899 non-null  float64
 17  lat     98899 non-null  float64
 18  区县      98899 non-null  float64
 19  板块      98899 non-null  float64
 20  环线位置    98899 non-null  float64
 21  建筑年代    72750 non-null  object 
 22

In [5]:
target='Price'
id_column=['ID']
text_columns_price=['房屋户型','所在楼层','交易时间','上次交易','年份','建筑年代','lon','lat','区域','板块','楼层类型']
columns_to_drop_price = text_columns_price + [target]

text_columns_rent=['户型','交易时间','配套设施','lon','lat','建筑年代','用水','用电','采暖','供水','供电',
                   '供暖','租期_短期','租期_长期','租期_中期']
columns_to_drop_rent = text_columns_rent + [target]

# 随机生成训练集和测试集
from sklearn.model_selection import train_test_split
X_price_train, X_price_test= train_test_split(price_train, test_size=0.2, random_state=111)
X_rent_train, X_rent_test= train_test_split(rent_train, test_size=0.2, random_state=111)

#重新构建二手房的自变量和因变量
y_price_train=X_price_train[target]
y_price_test=X_price_test[target]

X_price_train = X_price_train.drop(columns=columns_to_drop_price)
X_price_test = X_price_test.drop(columns=columns_to_drop_price)

#重新构建租房的自变量和因变量
y_rent_train=X_rent_train[target]
y_rent_test=X_rent_test[target]

X_rent_train = X_rent_train.drop(columns=columns_to_drop_rent)
X_rent_test = X_rent_test.drop(columns=columns_to_drop_rent)

In [6]:
rent_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 57 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      98899 non-null  int64  
 1   户型      98898 non-null  object 
 2   装修      98899 non-null  float64
 3   楼层      98899 non-null  float64
 4   面积      98899 non-null  float64
 5   交易时间    98899 non-null  object 
 6   付款方式    98899 non-null  int64  
 7   租赁方式    98899 non-null  int64  
 8   电梯      98899 non-null  float64
 9   车位      98899 non-null  int64  
 10  用水      90203 non-null  object 
 11  用电      90410 non-null  object 
 12  燃气      98899 non-null  float64
 13  采暖      40992 non-null  object 
 14  租期      98899 non-null  int64  
 15  配套设施    68448 non-null  object 
 16  lon     98899 non-null  float64
 17  lat     98899 non-null  float64
 18  区县      98899 non-null  float64
 19  板块      98899 non-null  float64
 20  环线位置    98899 non-null  float64
 21  建筑年代    72750 non-null  object 
 22

In [7]:
class SmartFeatureEngineer:
    def __init__(self):
        self.encoders = {}
        self.feature_mapping = {}
        self.training_columns = None
        self.fitted = False
    
    def log_transform(self, df, columns):
        """对数变换"""
        df = df.copy()
        for col in columns:
            if col in df.columns:
                min_val = df[col].min()
                shift = 1 - min_val if min_val <= 0 else 0
                new_col = f'log_{col}'
                df[new_col] = np.log(df[col] + shift)
                self.feature_mapping[new_col] = {'original': col, 'transformation': 'log', 'shift': shift}
        return df
    
    def binning(self, df, columns, n_bins=5):
        """分箱处理"""
        df = df.copy()
        for col in columns:
            if col in df.columns:
                binner = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
                new_col = f'{col}_bin'
                df[new_col] = binner.fit_transform(df[[col]]).astype(int).flatten()
                self.encoders[f'bin_{col}'] = binner
                self.feature_mapping[new_col] = {'original': col, 'transformation': 'binning', 'n_bins': n_bins}
        return df
    
    def polynomials(self, df, columns, degree=2):
        """多项式特征"""
        if not columns: return df
        
        existing_cols = [col for col in columns if col in df.columns]
        if not existing_cols: return df
        
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        poly_features = poly.fit_transform(df[existing_cols])
        feature_names = poly.get_feature_names_out(existing_cols)
        
        # 只添加新列
        new_cols = [col for col in feature_names if col not in df.columns]
        poly_df = pd.DataFrame(poly_features, columns=feature_names, index=df.index)[new_cols]
        
        for col in new_cols:
            self.feature_mapping[col] = {'original': existing_cols, 'transformation': 'polynomial'}
        
        return pd.concat([df, poly_df], axis=1)
    
    def interactions(self, df, pairs):
        """交互特征"""
        df = df.copy()
        for col1, col2 in pairs:
            if col1 in df.columns and col2 in df.columns:
                new_col = f'{col1}_x_{col2}'
                df[new_col] = df[col1] * df[col2]
                self.feature_mapping[new_col] = {'original': [col1, col2], 'transformation': 'interaction'}
        return df
    
    def dummies(self, df, columns=None, drop_first=True):
        """虚拟变量"""
        if columns is None:
            columns = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
        
        df_result = df.copy()
        
        for col in columns:
            if col in df.columns:
                if df[col].dtype == 'bool':
                    # 布尔列直接转换
                    new_col = f'{col}_num'
                    df_result[new_col] = df[col].astype(int)
                    self.feature_mapping[new_col] = {'original': col, 'transformation': 'bool_to_numeric'}
                else:
                    # 分类列创建虚拟变量
                    dummies = pd.get_dummies(df[col], prefix=col, drop_first=drop_first).astype(int)
                    for dummy_col in dummies.columns:
                        self.feature_mapping[dummy_col] = {'original': col, 'transformation': 'dummy'}
                    df_result = pd.concat([df_result, dummies], axis=1)
                    df_result = df_result.drop(columns=[col])  # 删除原始列
        
        return df_result
    
    def ratios(self, df, pairs):
        """比率特征"""
        df = df.copy()
        for num, denom in pairs:
            if num in df.columns and denom in df.columns:
                new_col = f'{num}_div_{denom}'
                # 使用更健壮的方法处理除零和NaN
                condition = df[denom] != 0
                df[new_col] = np.where(condition, df[num] / df[denom], 0)
                # 处理可能出现的无穷大值
                df[new_col] = df[new_col].replace([np.inf, -np.inf], 0)
                self.feature_mapping[new_col] = {'original': [num, denom], 'transformation': 'ratio'}
        return df
    
    def fit_transform(self, df, config):
        """训练集特征工程"""
        self.feature_mapping = {}
        df_transformed = df.copy()
        
        # 按配置执行特征工程
        if 'log' in config:
            df_transformed = self.log_transform(df_transformed, config['log'])
        
        if 'bins' in config:
            df_transformed = self.binning(df_transformed, config['bins']['columns'], 
                                        config['bins'].get('n_bins', 5))
        
        if 'poly' in config:
            df_transformed = self.polynomials(df_transformed, config['poly']['columns'], 
                                            config['poly'].get('degree', 2))
        
        if 'interactions' in config:
            df_transformed = self.interactions(df_transformed, config['interactions'])
        
        if 'dummies' in config:
            df_transformed = self.dummies(df_transformed, config['dummies'].get('columns'),
                                        config['dummies'].get('drop_first', True))
        
        if 'ratios' in config:
            df_transformed = self.ratios(df_transformed, config['ratios'])
        
        self.training_columns = df_transformed.columns.tolist()
        self.fitted = True
        
        print(f"特征工程完成: {len(df.columns)} -> {len(df_transformed.columns)} 个特征")
        return df_transformed
    
    def transform(self, df):
        """测试集特征工程"""
        if not self.fitted:
            raise ValueError("请先在训练集上调用 fit_transform")
        
        df_transformed = df.copy()
        
        # 应用训练集的变换
        for new_col, mapping in self.feature_mapping.items():
            transformation = mapping['transformation']
            original = mapping['original']
            
            if transformation == 'log' and original in df.columns:
                shift = mapping.get('shift', 0)
                df_transformed[new_col] = np.log(df[original] + shift)
            
            elif transformation == 'binning' and original in df.columns:
                binner = self.encoders.get(f'bin_{original}')
                if binner:
                    df_transformed[new_col] = binner.transform(df[[original]]).astype(int).flatten()
            
            elif transformation == 'interaction' and all(col in df.columns for col in original):
                df_transformed[new_col] = df[original[0]] * df[original[1]]
            
            elif transformation == 'ratio' and all(col in df.columns for col in original):
                condition = df[original[1]] != 0
                df_transformed[new_col] = np.where(condition, df[original[0]] / df[original[1]], 0)
                df_transformed[new_col] = df_transformed[new_col].replace([np.inf, -np.inf], 0)
            
            elif transformation == 'bool_to_numeric' and original in df.columns:
                df_transformed[new_col] = df[original].astype(int)

            
            elif transformation == 'dummy' and original in df.columns:
                # 为测试集创建虚拟变量
                dummies = pd.get_dummies(df[original], prefix=original).astype(int)
                if new_col in dummies.columns:
                    df_transformed[new_col] = dummies[new_col]
                else:
                    df_transformed[new_col] = 0  # 测试集没有的类别用0填充
        
        # 确保列顺序一致，缺失列用0填充
        missing_cols = set(self.training_columns) - set(df_transformed.columns)
        for col in missing_cols:
            df_transformed[col] = 0
        
        return df_transformed.reindex(columns=self.training_columns, fill_value=0)
    
    def get_feature_info(self):
        """获取特征信息"""
        return {
            'feature_mapping': self.feature_mapping,
            'training_columns': self.training_columns,
            'generated_features': list(self.feature_mapping.keys())
        }

In [8]:
#对二手房数据进行特征工程处理
config_X_price_train_processed = {
        'log': ['建筑面积','房屋总数'],
        'bins': {'columns': ['停车费用','小区平均房龄'], 'n_bins': 4},
        'bins': {'columns': ['地理聚类'], 'n_bins': 20},
        'dummies':{'columns':['配备电梯']},
        'poly': {'columns': ['卧室数','卫生间数','停车费用','客厅数','燃气费','物业费','绿化率','容积率','停车位'], 'degree': 2}, 
        'interactions': [('开发商', '停车费用'),('是否别墅', '建筑面积'),('建筑面积','物业费'),('物业费','物业公司'),
                         ('卫生间数','卧室数'),('配备电梯','总楼层')],
        'ratios':[('套内面积','建筑面积'),('房屋总数','楼栋总数'),('停车位','房屋总数'),('卫生间数', '卧室数'),
                ('绿化率','容积率')]
    }
engineer = SmartFeatureEngineer()
X_price_train_processed = engineer.fit_transform(X_price_train, config_X_price_train_processed)
X_price_test_processed = engineer.transform(X_price_test)
price_test_processed = engineer.fit_transform(price_test, config_X_price_train_processed)

y_price_train_processed=np.log(y_price_train)
y_price_test_processed=np.log(y_price_test)



特征工程完成: 48 -> 107 个特征
特征工程完成: 60 -> 119 个特征


In [9]:
#对租房数据进行特征工程处理
config_X_rent_train_processed = {
        'log': ['面积','房屋总数'],
        'bins': {'columns': ['停车费用','小区平均房龄','绿化率'], 'n_bins': 4},
        'bins': {'columns': ['地理聚类'], 'n_bins': 20},
        'poly': {'columns': ['卧室数','卫生间数','停车费用','物业费','容积率','配备设施数量'], 'degree': 2}, 
        'interactions': [('开发商', '停车费用'),('面积','物业费'),('物业费','物业公司'),('卫生间数','卧室数')
                        ,('电梯','楼层'),('装修','开发商')],
        'ratios':[('房屋总数','楼栋总数'),('停车位','房屋总数'),('绿化率','容积率'),('卫生间数', '卧室数')]
    }
engineer = SmartFeatureEngineer()
X_rent_train_processed = engineer.fit_transform(X_rent_train, config_X_rent_train_processed)
X_rent_test_processed = engineer.transform(X_rent_test)
rent_test_processed = engineer.fit_transform(rent_test, config_X_rent_train_processed)

y_rent_train_processed=np.log(y_rent_train)
y_rent_test_processed=np.log(y_rent_test)

特征工程完成: 41 -> 69 个特征
特征工程完成: 57 -> 85 个特征


In [10]:
class MultiTargetFeatureSelector:
    def __init__(self):
        self.selected_features = []
        self.lasso_models = {}
        
    def correlation_filter(self, df, target_cols, threshold=0.8):
        """基于多目标变量的相关性筛选特征"""
        # 计算与所有目标变量的平均相关性
        corr_with_targets = df[target_cols].apply(
            lambda target: df.corrwith(target).abs()
        ).mean(axis=1)
        
        # 移除目标变量自身
        corr_with_targets = corr_with_targets.drop(target_cols)
        
        # 计算特征间的相关性矩阵
        feature_cols = [col for col in df.columns if col not in target_cols]
        corr_matrix = df[feature_cols].corr().abs()
        
        # 选择高相关特征
        high_corr_features = corr_with_targets[corr_with_targets > 0.1].index.tolist()
        
        # 移除高度相关的特征
        upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper_triangle.columns 
                  if any(upper_triangle[column] > threshold)]
        
        selected = [col for col in high_corr_features if col not in to_drop]
        
        print(f"多目标相关性筛选: 从 {len(feature_cols)} 个特征中选择了 {len(selected)} 个特征")
        return selected
    
    def lasso_filter(self, df, features, target_cols, alpha=None):
        """基于LASSO的多目标特征筛选"""
        selected_features = set()
        
        # 对每个目标变量分别进行LASSO筛选
        for target in target_cols:
            print(f"\n对目标变量 '{target}' 进行LASSO筛选:")
            
            # 准备数据
            X = df[features].copy()
            y = df[target].copy()
            
            # 标准化特征（LASSO对特征尺度敏感）
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            
            # 使用交叉验证选择最佳alpha
            if alpha is None:
                lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
                lasso.fit(X_scaled, y)
                best_alpha = lasso.alpha_
                print(f"  交叉验证选择的最佳alpha: {best_alpha:.6f}")
            else:
                lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
                lasso.fit(X_scaled, y)
                best_alpha = alpha
            
            # 获取非零系数特征
            coef = lasso.coef_
            selected_indices = np.where(coef != 0)[0]
            target_selected_features = [features[i] for i in selected_indices]
            
            print(f"  选择了 {len(target_selected_features)} 个特征")
            if target_selected_features:
                print(f"  选中的特征: {target_selected_features}")
            
            # 保存模型
            self.lasso_models[target] = lasso
            
            # 添加到总特征集
            selected_features.update(target_selected_features)
        
        return list(selected_features)
    
    def comprehensive_selection(self, df, target_cols, corr_thresh=0.8, lasso_alpha=None):
        """综合特征选择方法：先相关性过滤，再LASSO过滤"""
        print("开始综合特征选择...")
        
        # 第一步：相关性过滤
        corr_selected = self.correlation_filter(df, target_cols, threshold=corr_thresh)
        
        if not corr_selected:
            print("警告：相关性过滤没有选择到任何特征，使用所有非目标特征")
            corr_selected = [col for col in df.columns if col not in target_cols]
        
        # 第二步：LASSO过滤
        lasso_selected = self.lasso_filter(df, corr_selected, target_cols, alpha=lasso_alpha)
        
        # 保存最终选择的特征
        self.selected_features = lasso_selected
        
        print(f"\n最终特征选择结果:")
        print(f"从 {len([col for col in df.columns if col not in target_cols])} 个候选特征中选择了 {len(lasso_selected)} 个特征")
        print(f"选中的特征: {lasso_selected}")
        
        return lasso_selected

In [11]:
# 创建特征选择器实例
selector = MultiTargetFeatureSelector()

# 对二手房执行特征选择（相关性和LASSO组合）
target_cols=['log_建筑面积', '停车费用^2','是否别墅','梯户比例','户型评分','相对楼层位置','卧室数^2','小区平均房龄',
             '套内面积_div_建筑面积','停车位_div_房屋总数','建筑面积_x_物业费','卫生间数_div_卧室数']
selected_features = selector.comprehensive_selection(
    X_price_train_processed, 
    target_cols,  # 您的目标变量
    corr_thresh=0.8,     # 相关性阈值
    lasso_alpha=None     # 让LASSO自动选择最佳alpha，也可以手动指定如0.01
)

# 获取筛选后的数据
X_price_train_processed = X_price_train_processed[selected_features + target_cols]
X_price_test_processed = X_price_test_processed[selected_features + target_cols]
price_test_processed= price_test_processed[selected_features + target_cols +id_column]

print(f"\n训练集形状: {X_price_train_processed.shape}")
print(f"测试集形状: {X_price_test_processed.shape}")

开始综合特征选择...
多目标相关性筛选: 从 95 个特征中选择了 15 个特征

对目标变量 'log_建筑面积' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000406
  选择了 13 个特征
  选中的特征: ['建筑面积', '别墅类型', '房屋用途', '供热费', '卧室数', '客厅数', '厨房数', '是否为地下室', '卧室数 容积率', '停车费用 停车位', '开发商_x_停车费用', '是否别墅_x_建筑面积', '房屋总数_div_楼栋总数']

对目标变量 '停车费用^2' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 1143.511090
  选择了 13 个特征
  选中的特征: ['建筑面积', '别墅类型', '房屋用途', '供热费', '卧室数', '客厅数', '厨房数', '是否为地下室', '停车费用 停车位', '客厅数 容积率', '物业费 停车位', '开发商_x_停车费用', '房屋总数_div_楼栋总数']

对目标变量 '是否别墅' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000110
  选择了 14 个特征
  选中的特征: ['建筑面积', '别墅类型', '房屋用途', '供热费', '卧室数', '客厅数', '厨房数', '是否为地下室', '卧室数 容积率', '停车费用 停车位', '物业费 停车位', '开发商_x_停车费用', '是否别墅_x_建筑面积', '房屋总数_div_楼栋总数']

对目标变量 '梯户比例' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000110
  选择了 14 个特征
  选中的特征: ['建筑面积', '别墅类型', '房屋用途', '供热费', '卧室数', '客厅数', '厨房数', '是否为地下室', '卧室数 容积率', '停车费用 停车位', '物业费 停车位', '开发商_x_停车费用', '是否别墅_x_建筑面积', '房屋总数_div_楼栋总数']

对目标变量 '户型评分' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000705
  选择了 11 个特征
  选中的特征: ['建筑面积', '别墅类型', '房屋用途', '供热费', '

In [12]:
# 对租房执行特征选择（相关性和LASSO组合）
target_cols=['log_面积', '户型评分','配套设施数量','电梯','租期','小区平均房龄','面积_x_物业费','停车费用^2','卧室数^2',
            '停车位_div_房屋总数']
selected_features = selector.comprehensive_selection(
    X_rent_train_processed, 
    target_cols,  # 您的目标变量
    corr_thresh=0.8,     # 相关性阈值
    lasso_alpha=None    # 让LASSO自动选择最佳alpha，也可以手动指定如0.01
)

# 获取筛选后的数据
X_rent_train_processed = X_rent_train_processed[selected_features +target_cols]
X_rent_test_processed = X_rent_test_processed[selected_features + target_cols]
rent_test_processed= rent_test_processed[selected_features + target_cols +id_column]


print(f"\n训练集形状: {X_rent_train_processed.shape}")
print(f"测试集形状: {X_rent_test_processed.shape}")

开始综合特征选择...
多目标相关性筛选: 从 59 个特征中选择了 16 个特征

对目标变量 'log_面积' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000533
  选择了 16 个特征
  选中的特征: ['面积', '租赁方式', '物业费', '供热费', '停车位', '停车费用', '卧室数', '客厅数', '设施是否完善', '采暖_合并', '卧室数 容积率', '卫生间数 停车费用', '卫生间数 物业费', '开发商_x_停车费用', '电梯_x_楼层', '房屋总数_div_楼栋总数']

对目标变量 '户型评分' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000736
  选择了 13 个特征
  选中的特征: ['面积', '租赁方式', '物业费', '供热费', '停车位', '停车费用', '卧室数', '客厅数', '设施是否完善', '卫生间数 停车费用', '卫生间数 物业费', '电梯_x_楼层', '房屋总数_div_楼栋总数']

对目标变量 '配套设施数量' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.003015
  选择了 14 个特征
  选中的特征: ['面积', '租赁方式', '物业费', '供热费', '停车位', '卧室数', '设施是否完善', '采暖_合并', '卧室数 容积率', '卫生间数 停车费用', '卫生间数 物业费', '开发商_x_停车费用', '电梯_x_楼层', '房屋总数_div_楼栋总数']

对目标变量 '电梯' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000400
  选择了 16 个特征
  选中的特征: ['面积', '租赁方式', '物业费', '供热费', '停车位', '停车费用', '卧室数', '客厅数', '设施是否完善', '采暖_合并', '卧室数 容积率', '卫生间数 停车费用', '卫生间数 物业费', '开发商_x_停车费用', '电梯_x_楼层', '房屋总数_div_楼栋总数']

对目标变量 '租期' 进行LASSO筛选:
  交叉验证选择的最佳alpha: 0.000280
  选择了 16 个特征
  选中的特征: ['面积', '租赁方式', '物

In [13]:
class OutlierProcessor:
    def __init__(self, categorical_keywords=None):
        self.iqr_bounds = {}
        self.zscore_params = {}  # 存储每列的均值和标准差
        self.categorical_keywords = categorical_keywords or ['cat', 'type', 'category', 'class', 'flag', 'bin', 'is_', 'has_', 'dummy']
    
    def set_categorical_keywords(self, categorical_keywords):
        """设置用于识别类别变量的关键词"""
        self.categorical_keywords = categorical_keywords
    
    def is_categorical_column(self, column_name):
        """检查列名是否包含类别变量的关键词"""
        column_lower = column_name.lower()
        return any(keyword in column_lower for keyword in self.categorical_keywords)
    
    def get_columns_to_process(self, df, columns=None, skip_categorical=True):
        """获取需要处理的数值列，跳过二值变量和类别变量"""
        if columns is None:
            columns = df.select_dtypes(include=[np.number]).columns
        
        columns_to_process = list(columns)
        
        # 跳过类别变量（基于关键词匹配）
        if skip_categorical:
            categorical_cols = [col for col in columns_to_process if self.is_categorical_column(col)]
            columns_to_process = [col for col in columns_to_process if col not in categorical_cols]
            if categorical_cols:
                print(f"跳过类别变量: {categorical_cols}")
        
        return columns_to_process
        
    def detect_outliers_iqr(self, df, columns=None, skip_categorical=True):
        """使用IQR方法检测异常值"""
        columns_to_process = self.get_columns_to_process(df, columns,  skip_categorical)
        
        outlier_info = {}
        for col in columns_to_process:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            outlier_info[col] = {
                'bounds': (lower_bound, upper_bound),
                'outlier_count': len(outliers),
                'outlier_percentage': len(outliers) / len(df) * 100
            }
            
            self.iqr_bounds[col] = (lower_bound, upper_bound)
        
        return pd.DataFrame(outlier_info).T
    
    def handle_outliers_iqr(self, df, columns=None, method='clip', skip_categorical=True):
        """
        使用IQR方法处理异常值
        method: 'clip'（截断）, 'remove'（删除）, 'transform'（变换）
        """
        columns_to_process = self.get_columns_to_process(df, columns, skip_categorical)
                
        df_processed = df.copy()
        report = {}
        
        for col in  columns_to_process:
            # 获取边界
            if col not in self.iqr_bounds:
                Q1, Q3 = df[col].quantile([0.25, 0.75])
                IQR = Q3 - Q1
                lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR
                self.iqr_bounds[col] = (lower, upper)
            else:
                lower, upper = self.iqr_bounds[col]
            
            original_len = len(df_processed)
            
            if method == 'clip':
                # 截断处理
                df_processed[col] = df_processed[col].clip(lower, upper)
                handled = ((df_processed[col] == lower) | (df_processed[col] == upper)).sum()
                
            elif method == 'remove':
                # 删除异常值
                mask = (df_processed[col] >= lower) & (df_processed[col] <= upper)
                df_processed = df_processed[mask]
                handled = original_len - len(df_processed)
                
            elif method == 'transform':
                # 对数变换
                if (df_processed[col] > 0).all():
                    df_processed[col] = np.log1p(df_processed[col])
                handled = 'N/A'
            
            report[col] = {
                'method': method,
                'handled_count': handled,
                'remaining_samples': len(df_processed)
            }
        
        print(f"异常值处理完成 - 方法: IQR {method}")
        return df_processed, report

In [14]:
#加入额外特征的数据集
train_data_price = pd.concat([X_price_train_processed, y_price_train_processed], axis=1)
train_data_rent = pd.concat([X_rent_train_processed, y_rent_train_processed], axis=1)

#调用类
categorical_keywords1 = ['供水','供暖','供电','政策周期','建筑结构','开发商','物业公司','聚类层级','地理聚类','建筑结构_comm',
                               '交易权属','城市','环线','楼层位置','配备电梯','产权所属','是否别墅','是否为底层','是否为顶层','是否为地下室']
processor = OutlierProcessor(categorical_keywords=categorical_keywords1)


IQR_report = processor.detect_outliers_iqr(train_data_price.drop(columns=[target]))
print(IQR_report)

train_data_price_cleaned, IQR_progress_report =  processor.handle_outliers_iqr(train_data_price, method='clip')

# 重新分离
X_price_train_processed = train_data_price_cleaned.drop(columns=[target])
y_price_train_processed = train_data_price_cleaned[target]

print(f"原始训练集大小: {len(train_data_price)}")
print(f"清理后训练集大小: {len(X_price_train_processed)}")
print(f"删除的样本数量: {len(train_data_price)-len(X_price_train_processed) }")

跳过类别变量: ['是否为地下室', '开发商_x_停车费用', '是否别墅_x_建筑面积', '是否别墅', '相对楼层位置']
                                                   bounds outlier_count  \
房屋用途                                           (0.0, 0.0)          6246   
客厅数                                           (-0.5, 3.5)           113   
厨房数                                            (1.0, 1.0)          2673   
别墅类型                                           (2.0, 2.0)          1157   
供热费                         (-27.095, 56.257000000000005)             0   
停车费用 停车位                  (-453681.99999999994, 999409.2)          5578   
卧室数 容积率         (-0.9999999999999991, 13.399999999999999)          4011   
建筑面积                         (-0.8599999999999852, 190.5)          3269   
客厅数 容积率                                     (-1.25, 8.75)          2519   
房屋总数_div_楼栋总数   (-148.55952380952377, 369.82142857142856)          4005   
卧室数                                            (0.5, 4.5)          3662   
物业费 停车位           (-3899.800000000

In [15]:
#对租房的训练集数据进行处理
categorical_keywords2 = ['城市','装修','付款方式','电梯','车位','燃气','租期','环线位置','开发商','物业公司','建筑结构','是否商品房',
                     '设施是否完善','政策周期','地理聚类','用水_合并','用电_合并','采暖_合并']
processor = OutlierProcessor(categorical_keywords=categorical_keywords2)

IQR_report = processor.detect_outliers_iqr(train_data_rent.drop(columns=[target]))
print(IQR_report)

train_data_rent_cleaned, IQR_progress_report =  processor.handle_outliers_iqr(train_data_rent, method='clip')

# 重新分离
X_rent_train_processed = train_data_rent_cleaned.drop(columns=[target])
y_rent_train_processed = train_data_rent_cleaned[target]

print(f"原始训练集大小: {len(train_data_rent)}")
print(f"清理后训练集大小: {len(X_rent_train_processed)}")
print(f"删除的样本数量: {len(train_data_rent)-len(X_rent_train_processed) }")

跳过类别变量: ['电梯_x_楼层', '设施是否完善', '停车位', '开发商_x_停车费用', '采暖_合并', '电梯', '租期', '停车位_div_房屋总数']
                                                  bounds outlier_count  \
客厅数                                          (-0.5, 3.5)            26   
停车费用                                     (-225.0, 855.0)          7754   
物业费            (-0.7135000000000002, 5.1225000000000005)          4652   
租赁方式                                          (1.0, 1.0)          4700   
卫生间数 物业费                      (-2.4000000000000004, 4.0)          7009   
供热费             (0.5050000000000061, 47.696999999999996)           203   
卧室数 容积率                     (-4.265000000000001, 15.375)          2814   
面积                        (-23.98750000000001, 174.0725)          2065   
卫生间数 停车费用                                (-291.0, 485.0)          8162   
房屋总数_div_楼栋总数             (-198.875, 472.79166666666663)          5079   
卧室数                                          (-2.0, 6.0)            47   
log_面积           (2.8895

In [53]:
def run_house_price_analysis(X_price_train, y_train, X_price_test, y_test=None):
    """
    过程式房屋价格预测分析
    """
    # 存储所有结果的字典
    results = {}
    performance_data = []
    
    # 数据标准化
    print("正在进行数据标准化...")
    scaler = StandardScaler()
    numeric_features = X_price_train.select_dtypes(include=[np.number]).columns
    
    X_price_train_scaled = X_price_train.copy()
    X_price_test_scaled = X_price_test.copy()
    X_price_train_scaled[numeric_features] = scaler.fit_transform(X_price_train[numeric_features])
    X_price_test_scaled[numeric_features] = scaler.transform(X_price_test[numeric_features])
    
    # 1. OLS 模型
    print("\n" + "="*50)
    print("训练 OLS 模型...")
    print("="*50)
    
    ols_model = LinearRegression()
    ols_model.fit(X_price_train_scaled, y_train)
    
    # OLS 预测
    y_train_ols = ols_model.predict(X_price_train_scaled)
    y_test_ols = ols_model.predict(X_price_test_scaled)
    
    # OLS 评估指标
    ols_train_mae = mean_absolute_error(y_train, y_train_ols)
    ols_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_ols))
    
    ols_test_mae = mean_absolute_error(y_test, y_test_ols) if y_test is not None else None
    ols_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_ols)) if y_test is not None else None
    
    # OLS 交叉验证
    kf = KFold(n_splits=2, shuffle=True, random_state=111)
    ols_cv_mae = -cross_val_score(ols_model, X_price_train_scaled, y_train, 
                                 cv=kf, scoring='neg_mean_absolute_error').mean()
    ols_cv_rmse = np.sqrt(-cross_val_score(ols_model, X_price_train_scaled, y_train, 
                                          cv=kf, scoring='neg_mean_squared_error')).mean()
    kaggle_score_ols=100* (1-0.5 * (ols_test_rmse+ols_test_mae))

    results['OLS'] = {
        'model': ols_model,
        'train_predictions': y_train_ols,
        'test_predictions': y_test_ols,
        'train_mae': ols_train_mae,
        'train_rmse': ols_train_rmse,
        'test_mae': ols_test_mae,
        'test_rmse': ols_test_rmse,
        'cv_mae': ols_cv_mae,
        'cv_rmse': ols_cv_rmse,
        'kaggle_score':kaggle_score_ols
    }
    
    performance_data.append({
        'Model': 'OLS',
        'Train_MAE': f"{ols_train_mae:.4f}",
        'Test_MAE': f"{ols_test_mae:.4f}" if ols_test_mae is not None else "N/A",
        'CV_MAE': f"{ols_cv_mae:.4f}",
        'Train_RMSE': f"{ols_train_rmse:.4f}",
        'Test_RMSE': f"{ols_test_rmse:.4f}" if ols_test_rmse is not None else "N/A",
        'CV_RMSE': f"{ols_cv_rmse:.4f}",
        'Best_Params': 'N/A',
        'Kaggle Score':kaggle_score_ols
    })
    
    # 2. LASSO 模型（带超参数优化）- 调整参数范围
    print("\n" + "="*50)
    print("训练 LASSO 模型（带超参数优化）...")
    print("="*50)
    
    lasso_param_grid = {
        'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
        'max_iter': [2000, 5000],
        'selection': ['cyclic', 'random']
    }
    
    lasso_grid = GridSearchCV(
        Lasso(random_state=111, tol=0.001),
        lasso_param_grid,
        cv=2,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )
    
    lasso_grid.fit(X_price_train_scaled, y_train)
    lasso_best = lasso_grid.best_estimator_
    lasso_best_params = lasso_grid.best_params_
    
    print(f"LASSO 最佳参数: {lasso_best_params}")
    
    # LASSO 预测
    y_train_lasso = lasso_best.predict(X_price_train_scaled)
    y_test_lasso = lasso_best.predict(X_price_test_scaled)
    
    # LASSO 评估指标
    lasso_train_mae = mean_absolute_error(y_train, y_train_lasso)
    lasso_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_lasso))
    
    lasso_test_mae = mean_absolute_error(y_test, y_test_lasso) if y_test is not None else None
    lasso_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_lasso)) if y_test is not None else None
    
    # LASSO 交叉验证
    lasso_cv_mae = -lasso_grid.best_score_
    lasso_cv_rmse = np.sqrt(-cross_val_score(lasso_best, X_price_train_scaled, y_train, 
                                           cv=kf, scoring='neg_mean_squared_error')).mean()
    kaggle_score_lasso=100* (1-0.5 * (lasso_test_mae+lasso_test_rmse))

    results['LASSO'] = {
        'model': lasso_best,
        'train_predictions': y_train_lasso,
        'test_predictions': y_test_lasso,
        'train_mae': lasso_train_mae,
        'train_rmse': lasso_train_rmse,
        'test_mae': lasso_test_mae,
        'test_rmse': lasso_test_rmse,
        'cv_mae': lasso_cv_mae,
        'cv_rmse': lasso_cv_rmse,
        'best_params': lasso_best_params,
        'Kaggle Score':kaggle_score_lasso
    }
    
    performance_data.append({
        'Model': 'LASSO',
        'Train_MAE': f"{lasso_train_mae:.4f}",
        'Test_MAE': f"{lasso_test_mae:.4f}" if lasso_test_mae is not None else "N/A",
        'CV_MAE': f"{lasso_cv_mae:.4f}",
        'Train_RMSE': f"{lasso_train_rmse:.4f}",
        'Test_RMSE': f"{lasso_test_rmse:.4f}" if lasso_test_rmse is not None else "N/A",
        'CV_RMSE': f"{lasso_cv_rmse:.4f}",
        'Best_Params': str(lasso_best_params),
        'Kaggle Score':kaggle_score_lasso
    })
    
    # 3. Ridge 模型（带超参数优化）- 调整参数范围
    print("\n" + "="*50)
    print("训练 Ridge 模型（带超参数优化）...")
    print("="*50)
    
    ridge_param_grid = {
        'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
        'solver': ['auto', 'svd', 'cholesky', 'lsqr']
    }
    
    ridge_grid = GridSearchCV(
        Ridge(random_state=42),
        ridge_param_grid,
        cv=2,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )
    
    ridge_grid.fit(X_price_train_scaled, y_train)
    ridge_best = ridge_grid.best_estimator_
    ridge_best_params = ridge_grid.best_params_
    
    print(f"Ridge 最佳参数: {ridge_best_params}")
    
    # Ridge 预测
    y_train_ridge = ridge_best.predict(X_price_train_scaled)
    y_test_ridge = ridge_best.predict(X_price_test_scaled)
    
    # Ridge 评估指标
    ridge_train_mae = mean_absolute_error(y_train, y_train_ridge)
    ridge_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_ridge))
    
    ridge_test_mae = mean_absolute_error(y_test, y_test_ridge) if y_test is not None else None
    ridge_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_ridge)) if y_test is not None else None
    
    # Ridge 交叉验证
    ridge_cv_mae = -ridge_grid.best_score_
    ridge_cv_rmse = np.sqrt(-cross_val_score(ridge_best, X_price_train_scaled, y_train, 
                                           cv=kf, scoring='neg_mean_squared_error')).mean()
    kaggle_score_ridge=100* (1-0.5 * (ridge_test_mae+ridge_test_rmse))
    results['Ridge'] = {
        'model': ridge_best,
        'train_predictions': y_train_ridge,
        'test_predictions': y_test_ridge,
        'train_mae': ridge_train_mae,
        'train_rmse': ridge_train_rmse,
        'test_mae': ridge_test_mae,
        'test_rmse': ridge_test_rmse,
        'cv_mae': ridge_cv_mae,
        'cv_rmse': ridge_cv_rmse,
        'best_params': ridge_best_params,
        'Kaggle Score':kaggle_score_ridge
    }
    
    performance_data.append({
        'Model': 'Ridge',
        'Train_MAE': f"{ridge_train_mae:.4f}",
        'Test_MAE': f"{ridge_test_mae:.4f}" if ridge_test_mae is not None else "N/A",
        'CV_MAE': f"{ridge_cv_mae:.4f}",
        'Train_RMSE': f"{ridge_train_rmse:.4f}",
        'Test_RMSE': f"{ridge_test_rmse:.4f}" if ridge_test_rmse is not None else "N/A",
        'CV_RMSE': f"{ridge_cv_rmse:.4f}",
        'Best_Params': str(ridge_best_params),
        'Kaggle Score':kaggle_score_ridge
    })

    # 4. Random Forest 模型（带超参数优化）- 调整参数范围
    print("\n" + "="*50)
    print("训练 Random Forest 模型（带超参数优化）...")
    print("="*50)
    
    rf_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [15, 20, 25, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['auto', 'sqrt', 'log2']
    }
    
    rf_grid = GridSearchCV(
        RandomForestRegressor(random_state=42, n_jobs=-1),
        rf_param_grid,
        cv=2,  # 为了速度，使用较少的折数
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )
    
    rf_grid.fit(X_price_train_scaled, y_train)
    rf_best = rf_grid.best_estimator_
    rf_best_params = rf_grid.best_params_
    
    print(f"Random Forest 最佳参数: {rf_best_params}")
    
    # Random Forest 预测
    y_train_rf = rf_best.predict(X_price_train_scaled)
    y_test_rf = rf_best.predict(X_price_test_scaled)
    
    # Random Forest 评估指标
    rf_train_mae = mean_absolute_error(y_train, y_train_rf)
    rf_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_rf))
    
    rf_test_mae = mean_absolute_error(y_test, y_test_rf) if y_test is not None else None
    rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_rf)) if y_test is not None else None
    
    # Random Forest 交叉验证
    rf_cv_mae = -rf_grid.best_score_
    rf_cv_rmse = np.sqrt(-cross_val_score(rf_best, X_price_train_scaled, y_train, 
                                        cv=kf, scoring='neg_mean_squared_error')).mean()
    kaggle_score_rf=100* (1-0.5 * (rf_test_mae+rf_test_rmse))
    results['RandomForest'] = {
        'model': rf_best,
        'train_predictions': y_train_rf,
        'test_predictions': y_test_rf,
        'train_mae': rf_train_mae,
        'train_rmse': rf_train_rmse,
        'test_mae': rf_test_mae,
        'test_rmse': rf_test_rmse,
        'cv_mae': rf_cv_mae,
        'cv_rmse': rf_cv_rmse,
        'best_params': rf_best_params,
        'Kaggle Score':kaggle_score_rf

    }
    performance_data.append({
        'Model': 'Rf',
        'Train_MAE': f"{rf_train_mae:.4f}",
        'Test_MAE': f"{rf_test_mae:.4f}" if rf_test_mae is not None else "N/A",
        'CV_MAE': f"{rf_cv_mae:.4f}",
        'Train_RMSE': f"{rf_train_rmse:.4f}",
        'Test_RMSE': f"{rf_test_rmse:.4f}" if rf_test_rmse is not None else "N/A",
        'CV_RMSE': f"{rf_cv_rmse:.4f}",
        'Best_Params': str(rf_best_params),
        'Kaggle Score':kaggle_score_rf
    })
    
    

    # 创建性能比较表格
    performance_df = pd.DataFrame(performance_data)
    
    # 找到最佳模型
    best_model_name = None
    best_cv_mae = float('inf')
    
    for name, result in results.items():
        if result['cv_mae'] < best_cv_mae:
            best_cv_mae = result['cv_mae']
            best_model_name = name
    
    # 添加最佳模型标记
    performance_df['Is_Best'] = performance_df['Model'] == best_model_name
    
    return results, performance_df, best_model_name, scaler

def display_results(performance_df, results, best_model_name, y_test=None):
    """
    显示结果和性能比较
    """
    print("\n" + "="*100)
    print("模型性能比较表")
    print("="*100)
    numeric_columns = ['Train_MAE', 'Train_RMSE', 'Test_MAE', 'Test_RMSE', 'CV_MAE','Kaggle Score']
    
    for col in numeric_columns:
        if col in performance_df.columns:
            performance_df[col] = pd.to_numeric(performance_df[col], errors='coerce')
            
    table_data = []
    for _, row in performance_df.iterrows():
        # 计算In Sample平均值 (MAE和RMSE的平均)
        in_sample_avg = 0.5*(row['Train_MAE'] + row['Train_RMSE'])
        kaggle_score = 100 * (1 -0.5* (row['Test_MAE'] + row['Test_RMSE']) )
        
        table_data.append({
            'Model': row['Model'],
            'In Sample': in_sample_avg,
            'Out of Sample MAE': row['Test_MAE'],
            'Out of Sample RMSE': row['Test_RMSE'],
            'Cross-Validation MAE': row['CV_MAE'],
            'Cross-Validation RMSE': row['CV_RMSE'],
            'Kaggle Score': kaggle_score
        })
    performance_table = pd.DataFrame(table_data)

    display(performance_table)

        
    print(f"\n🎯 最佳模型: {best_model_name}")
    print(f"📊 最佳模型交叉验证 MAE: {results[best_model_name]['cv_mae']:.4f}")
    
    if y_test is not None:
        print(f"🧪 最佳模型测试集 MAE: {results[best_model_name]['test_mae']:.4f}")
    
    # 显示预测统计
    best_predictions = results[best_model_name]['test_predictions']
    print(f"\n📈 最佳模型预测统计 (log价格):")
    print(f"   最小值: {best_predictions.min():.2f}")
    print(f"   最大值: {best_predictions.max():.2f}")
    print(f"   平均值: {best_predictions.mean():.2f}")
    print(f"   中位数: {np.median(best_predictions):.2f}")
    print(f"   标准差: {best_predictions.std():.2f}")

def predict_new_data(results, best_model_name, scaler, X_new, id_column=None):
    """
    使用最佳模型预测新的数据集，并将log价格转换回正常价格
    
    参数:
    results: 训练结果字典
    best_model_name: 最佳模型名称
    scaler: 用于标准化的scaler对象
    X_new: 新的测试特征数据集
    id_column: ID列的名称，如果为None则自动生成ID
    """
    # 获取最佳模型
    best_model = results[best_model_name]['model']
    
    # 复制数据以避免修改原始数据
    X_new_processed = X_new.copy()
    if id_column is not None and id_column in X_new_processed.columns:
        ids = X_new_processed[id_column]
        # 移除ID列，因为模型训练时没有这个特征
        X_new_processed = X_new_processed.drop(columns=[id_column])
    else:
        # 如果没有指定ID列，自动生成从1开始的ID
        ids = range(1, len(X_new_processed) + 1)
    
    # 使用相同的scaler转换新数据
    numeric_features = X_new_processed.select_dtypes(include=[np.number]).columns
    X_new_processed[numeric_features] = scaler.transform(X_new_processed[numeric_features])
    
    # 预测log价格
    y_pred_log = best_model.predict(X_new_processed)
    
    # 将log价格转换回正常价格
    # 使用expm1来反向转换log(1+price)
    y_pred_original = np.expm1(y_pred_log)
    
    
    result_df = pd.DataFrame({
        'ID': ids,
        'Price': y_pred_original
    })
    
    return result_df, y_pred_log, y_pred_original


In [54]:
def run_house_analysis():
    """
    运行二手房价格预测分析
    """
    print("正在进行二手房模型训练和选择...")
    results, performance_df, best_model_name, scaler = run_house_price_analysis(
        X_price_train_processed, y_price_train_processed, X_price_test_processed, y_price_test_processed
    )
    
    # 显示结果
    display_results(performance_df, results, best_model_name, y_price_test_processed)
    
    # 预测二手房测试数据
    print("\n" + "="*80)
    print("开始预测二手房测试数据集...")
    print("="*80)
    
    # 预测二手房数据
    house_predictions, house_pred_log, house_pred_original = predict_new_data(
        results, best_model_name, scaler, 
        X_new=price_test_processed,  
        id_column='ID'          # 二手房数据的ID列名
    )
    
    # 显示二手房预测统计
    print(f"\n 二手房数据集预测统计:")
    print(f"   预测样本数量: {len(house_predictions)}")
    print(f"   预测价格最小值: ${house_pred_original.min():.2f}")
    print(f"   预测价格最大值: ${house_pred_original.max():.2f}")
    print(f"   预测价格平均值: ${house_pred_original.mean():.2f}")
    print(f"   预测价格中位数: ${np.median(house_pred_original):.2f}")
    
    return house_predictions

def run_rental_analysis():
    """
    运行租房价格预测分析
    """
    print("正在进行租房模型训练和选择...")
    
    results_rent, performance_df_rent, best_model_name_rent, scaler_rent = run_house_price_analysis(
        X_rent_train_processed, y_rent_train_processed, X_rent_test_processed, y_rent_test_processed
    )
    
    # 显示租房结果
    display_results(performance_df_rent, results_rent, best_model_name_rent, y_rent_test_processed)
    
    # 预测租房测试数据
    print("\n" + "="*80)
    print("开始预测租房测试数据集...")
    print("="*80)
    
    # 预测租房数据
    rental_predictions, rental_pred_log, rental_pred_original = predict_new_data(
        results_rent, best_model_name_rent, scaler_rent, 
        X_new=rent_test_processed,  # 替换为您的租房测试数据
        id_column='ID'         # 租房数据的ID列名
    )
    
    # 显示租房预测统计
    print(f"\n 租房数据集预测统计:")
    print(f"   预测样本数量: {len(rental_predictions)}")
    print(f"   预测价格最小值: ${rental_pred_original.min():.2f}")
    print(f"   预测价格最大值: ${rental_pred_original.max():.2f}")
    print(f"   预测价格平均值: ${rental_pred_original.mean():.2f}")
    print(f"   预测价格中位数: ${np.median(rental_pred_original):.2f}")
    
    return rental_predictions

def save_combined_results(house_predictions=None, rental_predictions=None, filename='combined_predictions.csv'):
    """
    将二手房和租房的预测结果合并到一个CSV文件中
    
    参数:
    house_predictions: 二手房预测结果DataFrame
    rental_predictions: 租房预测结果DataFrame
    filename: 输出文件名
    """
    # 创建空的DataFrame来存储所有结果
    all_predictions = pd.DataFrame(columns=['ID', 'Price'])
    
    # 添加二手房预测结果
    if house_predictions is not None:
        # 确保列名一致
        house_df = house_predictions.rename(columns={'ID': 'ID', 'Price': 'Price'})
        all_predictions = pd.concat([all_predictions, house_df], ignore_index=True)
        print(f"已添加 {len(house_df)} 条二手房预测记录")
    
    # 添加租房预测结果
    if rental_predictions is not None:
        # 确保列名一致
        rental_df = rental_predictions.rename(columns={'ID': 'ID', 'Price': 'Price'})
        all_predictions = pd.concat([all_predictions, rental_df], ignore_index=True)
        print(f"已添加 {len(rental_df)} 条租房预测记录")
    
    # 按ID排序（可选）
    all_predictions = all_predictions.sort_values('ID').reset_index(drop=True)
    
    # 保存到CSV
    all_predictions.to_csv(filename, index=False)
    print(f"\n💾 合并后的预测结果已保存到 '{filename}'")
    print(f"📊 总记录数: {len(all_predictions)}")
    
    # 显示前几行
    print(f"\n🔍 合并文件前5行预览:")
    print(all_predictions.head())
    
    return all_predictions

In [ ]:
# 主函数 - 可以选择运行二手房或租房分析
def main(dataset_type="both"):
    """
    主函数，可以选择运行二手房或租房分析
    
    参数:
    dataset_type: 'house' 运行二手房分析, 'rental' 运行租房分析, 'both' 运行两者
    """
    house_results = None
    rental_results = None
    
    if dataset_type in ["house", "both"]:
        print("开始二手房分析...")
        house_results = run_house_analysis()
        # 单独保存二手房结果
        house_results.to_csv('house_predictions.csv', index=False)
        print("二手房预测结果已保存到 'house_predictions.csv'")
    
    if dataset_type in ["rental", "both"]:
        print("\n 开始租房分析...")
        rental_results = run_rental_analysis()
        # 单独保存租房结果
        rental_results.to_csv('rental_predictions.csv', index=False)
        print(" 租房预测结果已保存到 'rental_predictions.csv'")
    
    # 如果两者都运行了，合并结果
    if dataset_type == "both" and house_results is not None and rental_results is not None:
        print("\n 合并二手房和租房预测结果...")
        combined_results = save_combined_results(house_results, rental_results)
        return house_results, rental_results, combined_results
    elif dataset_type == "house" and house_results is not None:
        return house_results
    elif dataset_type == "rental" and rental_results is not None:
        return rental_results


if __name__ == "__main__":
    house_predictions, rental_predictions, combined_predictions = main("both")
    
    print("\n✅ 所有分析完成！")

开始二手房分析...
正在进行二手房模型训练和选择...
正在进行数据标准化...

训练 OLS 模型...

训练 LASSO 模型（带超参数优化）...
Fitting 2 folds for each of 24 candidates, totalling 48 fits
